# 🔬 TensorFlow Experiments — Nitrate Nowcasting
> Estación Yeguas · 4 arquitecturas · Walk-Forward Validation · Integrated Gradients

**Arquitecturas comparadas**
| ID | Modelo |
|---|---|
| A | MLP Wide & Deep |
| B | GRU |
| C | 1D-CNN + LSTM |
| D | Transformer (Atención Temporal) |

## 0 · Setup & Cargar datos

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'      # silencia logs de C++ de TF
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'     # desactiva oneDNN (evita freeze)
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'     # forzar CPU si no hay GPU real

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers, regularizers
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mlflow
import mlflow.tensorflow
import joblib, json

print(f'TensorFlow {tf.__version__}')
print(f'MLflow     {mlflow.__version__}')
tf.random.set_seed(42)
np.random.seed(42)

# ── Paleta de colores por modelo ──────────────────────────────────────────────
COLORS = {'A_WideDeep': '#2196F3', 'B_GRU': '#4CAF50', 'C_CNN_LSTM': '#FF9800', 'D_Transformer': '#9C27B0'}
MODEL_NAMES = {'A_WideDeep': 'Wide & Deep', 'B_GRU': 'GRU', 'C_CNN_LSTM': '1D-CNN+LSTM', 'D_Transformer': 'Transformer'}

In [ ]:
# ── Configuración MLflow ──────────────────────────────────────────────────────
MLFLOW_EXPERIMENT = "nitrate_nowcasting"
MLFLOW_TRACKING_URI = "http://mlflow.example.com:5000"          # carpeta local; cambia a http://... para servidor remoto

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
print(f'Experimento : {MLFLOW_EXPERIMENT}')
print(f'Tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experiment ID: {exp.experiment_id}')
print()
print('Para visualizar los resultados, ejecuta en la terminal:')
print('  mlflow ui --port 5000')
print('  → http://127.0.0.1:5000')

In [ ]:
# ── Carga y limpieza ──────────────────────────────────────────────────────────
df = pd.read_csv('YEGUAS_data.csv')
df['_time'] = pd.to_datetime(df['_time'])
df = df.set_index('_time').sort_index()

# Eliminar columnas NITRATE excepto el target Y
cols_to_drop = [c for c in df.columns if 'NITRATE' in c and c != 'NITRATE_nitrate_no3_mg_l']
df = df.drop(columns=cols_to_drop)
df = df.dropna(subset=['NITRATE_nitrate_no3_mg_l'])
df = df.interpolate(method='linear').dropna()

print(f'Filas: {len(df)}  |  Features: {df.shape[1]-1}  |  Target: NITRATE_nitrate_no3_mg_l')
print(f'Rango: {df.index[0]}  →  {df.index[-1]}')
df.head(3)

## 1 · Preprocesado

In [ ]:
TARGET = 'NITRATE_nitrate_no3_mg_l'
FEATURE_COLS = [c for c in df.columns if c != TARGET]

# ── Hiperparámetros globales ──────────────────────────────────────────────────
WINDOW = 8          # pasos de 30 min = 4 h de historia (reducido vs 48 original)
LAG_STEPS = [1, 2, 4]  # lags para Wide & Deep

# ── Separar X e Y ─────────────────────────────────────────────────────────────
y_raw = df[TARGET].values
X_raw = df[FEATURE_COLS].values

# ── Escalado ──────────────────────────────────────────────────────────────────
X_scaler = StandardScaler()
X_scaled = X_scaler.fit_transform(X_raw)

y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).flatten()

print(f'X escalado: {X_scaled.shape}  |  y escalado: {y_scaled.shape}')

In [ ]:
# ── Función para crear ventanas temporales (modelos B/C/D) ────────────────────
def make_sequences(X, y, window):
    """Genera arrays 3D (n_samples, window, n_features) para modelos recurrentes."""
    Xs, ys = [], []
    for i in range(window - 1, len(X)):
        Xs.append(X[i - window + 1 : i + 1])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

# ── Función para crear lag features (modelo A Wide & Deep) ───────────────────
def make_lag_features(X, y, lags):
    """Añade columnas de lag. Retorna arrays 2D para MLP."""
    max_lag = max(lags)
    lag_arrays = [X[max_lag:]]
    for lag in lags:
        lag_arrays.append(X[max_lag - lag : len(X) - lag])
    X_with_lags = np.concatenate(lag_arrays, axis=1)
    y_aligned = y[max_lag:]
    return X_with_lags.astype(np.float32), y_aligned.astype(np.float32)

X_seq, y_seq = make_sequences(X_scaled, y_scaled, WINDOW)
X_lag, y_lag = make_lag_features(X_scaled, y_scaled, LAG_STEPS)

print(f'Secuencias  (B/C/D): {X_seq.shape}')
print(f'Lag features  (A):   {X_lag.shape}')

## 2 · Walk-Forward Validation — Definición de Folds

In [ ]:
def make_folds(n_samples, n_folds=5, min_train_ratio=0.6):
    """Genera índices para walk-forward expanding-window."""
    min_train = int(n_samples * min_train_ratio)
    remaining = n_samples - min_train
    fold_size = remaining // n_folds
    folds = []
    for k in range(n_folds):
        train_end = min_train + k * fold_size
        test_end  = train_end + fold_size
        if k == n_folds - 1:
            test_end = n_samples  # último fold usa todo lo que queda
        folds.append((slice(0, train_end), slice(train_end, test_end)))
    return folds

N_FOLDS = 5
folds_seq = make_folds(len(X_seq), N_FOLDS)
folds_lag = make_folds(len(X_lag), N_FOLDS)

print('Folds walk-forward (secuencias):')
for i, (tr, te) in enumerate(folds_seq):
    print(f'  Fold {i+1}: train[{tr.start}:{tr.stop}]  test[{te.start}:{te.stop}]')

## 3 · Modelo A — MLP Wide & Deep

In [ ]:
def build_wide_deep(input_dim):
    """
    Wide & Deep:
      Wide  ← features contemporáneas (explotar relación lineal)
      Deep  ← features contemporáneas + lags (no linealidades)
    """
    n_current = len(FEATURE_COLS)         # features del instante t
    n_lags    = input_dim - n_current      # features de lags

    inp = keras.Input(shape=(input_dim,), name='input')

    # ── Wide branch (solo features contemporáneas) ────────────────────────────
    wide = layers.Lambda(lambda x: x[:, :n_current], name='wide_slice')(inp)

    # ── Deep branch ───────────────────────────────────────────────────────────
    deep = layers.Dense(64, activation='relu',
                        kernel_regularizer=regularizers.l2(1e-4))(inp)
    deep = layers.BatchNormalization()(deep)
    deep = layers.Dropout(0.2)(deep)
    deep = layers.Dense(32, activation='relu',
                        kernel_regularizer=regularizers.l2(1e-4))(deep)

    # ── Fusión ────────────────────────────────────────────────────────────────
    merged = layers.Concatenate()([wide, deep])
    out = layers.Dense(16, activation='relu')(merged)
    out = layers.Dense(1)(out)

    model = keras.Model(inp, out, name='A_WideDeep')
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return model

# Preview de la arquitectura
build_wide_deep(X_lag.shape[1]).summary()

## 4 · Modelo B — GRU

In [ ]:
def build_gru(window, n_features):
    model = keras.Sequential([
        keras.Input(shape=(window, n_features)),
        layers.GRU(64, return_sequences=False,
                   kernel_regularizer=regularizers.l2(1e-4),
                   recurrent_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.2),
        layers.BatchNormalization(),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name='B_GRU')
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return model

build_gru(WINDOW, X_seq.shape[2]).summary()

## 5 · Modelo C — 1D-CNN + LSTM

In [ ]:
def build_cnn_lstm(window, n_features):
    inp = keras.Input(shape=(window, n_features))

    # ── CNN: detectar patrones locales (picos turbidez, variaciones rápidas) ──
    x = layers.Conv1D(32, kernel_size=3, activation='relu', padding='causal')(inp)
    x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='causal')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.1)(x)

    # ── LSTM: capturar dependencias temporales más largas ─────────────────────
    x = layers.LSTM(32, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation='relu')(x)
    out = layers.Dense(1)(x)

    model = keras.Model(inp, out, name='C_CNN_LSTM')
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return model

build_cnn_lstm(WINDOW, X_seq.shape[2]).summary()

## 6 · Modelo D — Transformer (Atención Temporal)

In [ ]:
def build_transformer(window, n_features, num_heads=4, key_dim=16, ff_dim=64, dropout=0.2):
    inp = keras.Input(shape=(window, n_features))

    # ── Proyección de entrada ──────────────────────────────────────────────────
    x = layers.Dense(key_dim * num_heads)(inp)   # proyectar a dimensión del modelo

    # ── Bloque Transformer ─────────────────────────────────────────────────────
    attn_out = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim, dropout=dropout)(x, x)
    x = layers.Add()([x, attn_out])             # residual
    x = layers.LayerNormalization()(x)

    # Feed-forward
    ff = layers.Dense(ff_dim, activation='relu')(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(key_dim * num_heads)(ff)
    x  = layers.Add()([x, ff])                 # residual
    x  = layers.LayerNormalization()(x)

    # ── Pooling y cabeza de regresión ──────────────────────────────────────────
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1)(x)

    model = keras.Model(inp, out, name='D_Transformer')
    model.compile(optimizer=keras.optimizers.Adam(5e-4), loss='mse', metrics=['mae'])
    return model

build_transformer(WINDOW, X_seq.shape[2]).summary()

## 7 · Walk-Forward Validation — Ejecución

In [ ]:
def run_walk_forward(model_fn, X, y, folds, build_kwargs, epochs=100, batch_size=32):
    """
    Ejecuta walk-forward validation para un model_fn dado.
    Si hay un mlflow.active_run(), registra métricas por fold (step) y resumen.
    Retorna DataFrame con métricas por fold y todas las predicciones.
    """
    results = []
    all_preds, all_true = [], []

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=0)

    for fold_idx, (tr_sl, te_sl) in enumerate(folds):
        X_tr, y_tr = X[tr_sl], y[tr_sl]
        X_te, y_te = X[te_sl], y[te_sl]

        model = model_fn(**build_kwargs)
        model.fit(
            X_tr, y_tr,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=0.15,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        y_pred_scaled = model.predict(X_te, verbose=0).flatten()
        y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        y_true = y_scaler.inverse_transform(y_te.reshape(-1, 1)).flatten()

        mae  = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2   = r2_score(y_true, y_pred)

        # ── MLflow: métricas por fold como pasos de la curva ──────────────────
        if mlflow.active_run():
            mlflow.log_metrics(
                {"fold_mae": float(mae), "fold_rmse": float(rmse), "fold_r2": float(r2)},
                step=fold_idx + 1
            )

        results.append({'Fold': fold_idx + 1, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
        all_preds.extend(y_pred)
        all_true.extend(y_true)
        print(f'  Fold {fold_idx+1}/{len(folds)}  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}')

    df_results = pd.DataFrame(results)

    # ── MLflow: métricas resumen del experimento ───────────────────────────────
    if mlflow.active_run():
        last3 = df_results.tail(3)
        mlflow.log_metrics({
            "mean_mae":   float(df_results['MAE'].mean()),
            "mean_rmse":  float(df_results['RMSE'].mean()),
            "mean_r2":    float(df_results['R2'].mean()),
            "last3_mae":  float(last3['MAE'].mean()),
            "last3_rmse": float(last3['RMSE'].mean()),
            "last3_r2":   float(last3['R2'].mean()),
        })

    return df_results, np.array(all_true), np.array(all_preds)

In [ ]:
print('='*60)
print('Modelo A — Wide & Deep')
print('='*60)

with mlflow.start_run(run_name="A_WideDeep", tags={"architecture": "MLP"}) as run_A:
    mlflow.log_params({
        "model_type":   "Wide_Deep_MLP",
        "input_type":   "lag_features",
        "lag_steps":    str(LAG_STEPS),
        "input_dim":    X_lag.shape[1],
        "wide_dim":     len(FEATURE_COLS),
        "dense_units":  "64-32-16",
        "dropout":      0.2,
        "l2_reg":       1e-4,
        "epochs":       100,
        "batch_size":   32,
        "n_folds":      N_FOLDS,
    })

    r_A, true_A, pred_A = run_walk_forward(
        model_fn=build_wide_deep,
        X=X_lag, y=y_lag,
        folds=folds_lag,
        build_kwargs={'input_dim': X_lag.shape[1]}
    )

    # Reentrenar con todos los datos y registrar modelo en MLflow
    _es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=0)
    final_A = build_wide_deep(X_lag.shape[1])
    final_A.fit(X_lag, y_lag, epochs=150, batch_size=32, validation_split=0.1, callbacks=[_es], verbose=0)
    mlflow.tensorflow.log_model(final_A, "model",
                                 registered_model_name=None)   # se registra el ganador al final
    run_id_A = run_A.info.run_id

print(r_A.to_string(index=False))
print(f'MLflow Run ID: {run_id_A}')

In [ ]:
print('='*60)
print('Modelo B — GRU')
print('='*60)

with mlflow.start_run(run_name="B_GRU", tags={"architecture": "RNN"}) as run_B:
    mlflow.log_params({
        "model_type":    "GRU",
        "input_type":    "sequences",
        "window":        WINDOW,
        "n_features":    X_seq.shape[2],
        "gru_units":     64,
        "dropout":       0.2,
        "l2_reg":        1e-4,
        "batch_norm":    True,
        "dense_units":   32,
        "epochs":        100,
        "batch_size":    32,
        "n_folds":       N_FOLDS,
    })

    r_B, true_B, pred_B = run_walk_forward(
        model_fn=build_gru,
        X=X_seq, y=y_seq,
        folds=folds_seq,
        build_kwargs={'window': WINDOW, 'n_features': X_seq.shape[2]}
    )

    _es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=0)
    final_B = build_gru(WINDOW, X_seq.shape[2])
    final_B.fit(X_seq, y_seq, epochs=150, batch_size=32, validation_split=0.1, callbacks=[_es], verbose=0)
    mlflow.tensorflow.log_model(final_B, "model")
    run_id_B = run_B.info.run_id

print(r_B.to_string(index=False))
print(f'MLflow Run ID: {run_id_B}')

In [ ]:
print('='*60)
print('Modelo C — 1D-CNN + LSTM')
print('='*60)

with mlflow.start_run(run_name="C_CNN_LSTM", tags={"architecture": "CNN+RNN"}) as run_C:
    mlflow.log_params({
        "model_type":    "1D_CNN_LSTM",
        "input_type":    "sequences",
        "window":        WINDOW,
        "n_features":    X_seq.shape[2],
        "conv_filters":  "32-64",
        "conv_kernel":   3,
        "conv_padding":  "causal",
        "lstm_units":    32,
        "dropout":       0.2,
        "l2_reg":        1e-4,
        "epochs":        100,
        "batch_size":    32,
        "n_folds":       N_FOLDS,
    })

    r_C, true_C, pred_C = run_walk_forward(
        model_fn=build_cnn_lstm,
        X=X_seq, y=y_seq,
        folds=folds_seq,
        build_kwargs={'window': WINDOW, 'n_features': X_seq.shape[2]}
    )

    _es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=0)
    final_C = build_cnn_lstm(WINDOW, X_seq.shape[2])
    final_C.fit(X_seq, y_seq, epochs=150, batch_size=32, validation_split=0.1, callbacks=[_es], verbose=0)
    mlflow.tensorflow.log_model(final_C, "model")
    run_id_C = run_C.info.run_id

print(r_C.to_string(index=False))
print(f'MLflow Run ID: {run_id_C}')

In [ ]:
print('='*60)
print('Modelo D — Transformer')
print('='*60)

with mlflow.start_run(run_name="D_Transformer", tags={"architecture": "Attention"}) as run_D:
    mlflow.log_params({
        "model_type":    "Transformer",
        "input_type":    "sequences",
        "window":        WINDOW,
        "n_features":    X_seq.shape[2],
        "num_heads":     4,
        "key_dim":       16,
        "ff_dim":        64,
        "dropout":       0.2,
        "l2_reg":        1e-4,
        "learning_rate": 5e-4,
        "epochs":        100,
        "batch_size":    32,
        "n_folds":       N_FOLDS,
    })

    r_D, true_D, pred_D = run_walk_forward(
        model_fn=build_transformer,
        X=X_seq, y=y_seq,
        folds=folds_seq,
        build_kwargs={'window': WINDOW, 'n_features': X_seq.shape[2]}
    )

    _es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=0)
    final_D = build_transformer(WINDOW, X_seq.shape[2])
    final_D.fit(X_seq, y_seq, epochs=150, batch_size=32, validation_split=0.1, callbacks=[_es], verbose=0)
    mlflow.tensorflow.log_model(final_D, "model")
    run_id_D = run_D.info.run_id

print(r_D.to_string(index=False))
print(f'MLflow Run ID: {run_id_D}')

## 8 · Tabla comparativa y visualización

In [ ]:
# ── Consultar MLflow para obtener todos los runs del experimento ───────────────
runs_df = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    order_by=["metrics.last3_mae ASC"],        # orden: mejor primero
)

display_cols = [
    'tags.mlflow.runName',
    'metrics.mean_mae',  'metrics.mean_rmse',  'metrics.mean_r2',
    'metrics.last3_mae', 'metrics.last3_rmse', 'metrics.last3_r2',
    'run_id',
]
comparison_mlflow = runs_df[display_cols].copy()
comparison_mlflow.columns = [
    'Modelo',
    'MAE medio', 'RMSE medio', 'R² medio',
    'MAE últimos 3', 'RMSE últimos 3', 'R² últimos 3',
    'run_id',
]
comparison_mlflow = comparison_mlflow.set_index('Modelo')

print('\n── Tabla Comparativa Walk-Forward (desde MLflow) ──')
print(comparison_mlflow.drop(columns='run_id').to_string())

# ── Ganador: menor MAE en los últimos 3 folds ─────────────────────────────────
best_run_row  = runs_df.iloc[0]
best_run_id   = best_run_row['run_id']
best_run_name = best_run_row['tags.mlflow.runName']

# Mapear nombre MLflow → clave interna
_name_map = {v: k for k, v in MODEL_NAMES.items()}  # 'GRU' → 'B_GRU'
best_name = _name_map.get(
    best_run_name.split('_', 1)[-1].replace('_', ' ') if '_' in best_run_name else best_run_name,
    best_run_name
)
# fallback directo si el run_name ya es la clave ('A_WideDeep', 'B_GRU', ...)
if best_run_name in MODEL_NAMES:
    best_name = best_run_name

print(f'\n🏆 Modelo ganador : {best_run_name}')
print(f'   MAE últimos 3  : {best_run_row["metrics.last3_mae"]:.3f} mg/L')
print(f'   R² últimos 3   : {best_run_row["metrics.last3_r2"]:.4f}')
print(f'   Run ID         : {best_run_id}')
print(f'\n   mlflow ui → http://127.0.0.1:5000  (pestaña Experiments → {MLFLOW_EXPERIMENT})')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

datasets = [
    ('A_WideDeep',   r_A, true_A, pred_A),
    ('B_GRU',        r_B, true_B, pred_B),
    ('C_CNN_LSTM',   r_C, true_C, pred_C),
    ('D_Transformer',r_D, true_D, pred_D),
]

for ax, (name, results, y_true, y_pred) in zip(axes, datasets):
    color = COLORS[name]
    ax.plot(y_true[-200:],  label='Real',     color='#333', linewidth=1.5, alpha=0.8)
    ax.plot(y_pred[-200:],  label='Predicho', color=color,  linewidth=1.5, alpha=0.9, linestyle='--')
    mae_last = results['MAE'].tail(3).mean()
    r2_last  = results['R2'].tail(3).mean()
    ax.set_title(f'{MODEL_NAMES[name]}\nMAE={mae_last:.3f} mg/L  R²={r2_last:.4f}', fontsize=11)
    ax.set_xlabel('Muestra (últimas 200)')
    ax.set_ylabel('NO₃⁻ (mg/L)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparativa TensorFlow — Nitrate Nowcasting\nEstación Yeguas', fontsize=14, y=1.02)
plt.tight_layout()
_plot_path = 'model_comparison.png'
plt.savefig(_plot_path, dpi=150, bbox_inches='tight')
plt.show()

# ── Registrar gráfico como artefacto en el run ganador ────────────────────────
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(_plot_path, artifact_path="plots")
print(f'Gráfico guardado y registrado en MLflow (run: {best_run_name})')

In [ ]:
# Evolución de MAE por fold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, results in [('A_WideDeep', r_A), ('B_GRU', r_B), ('C_CNN_LSTM', r_C), ('D_Transformer', r_D)]:
    axes[0].plot(results['Fold'], results['MAE'],   marker='o', label=MODEL_NAMES[name], color=COLORS[name])
    axes[1].plot(results['Fold'], results['R2'],    marker='o', label=MODEL_NAMES[name], color=COLORS[name])

for ax, title, ylabel in [
    (axes[0], 'MAE por Fold', 'MAE (mg/L)'),
    (axes[1], 'R² por Fold',  'R²'),
]:
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Fold')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(1, N_FOLDS + 1))

plt.suptitle('Walk-Forward Validation — Evolución por Fold', fontsize=13)
plt.tight_layout()
_folds_plot_path = 'walk_forward_folds.png'
plt.savefig(_folds_plot_path, dpi=150, bbox_inches='tight')
plt.show()

# ── Registrar también este gráfico ────────────────────────────────────────────
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(_folds_plot_path, artifact_path="plots")
print('Gráfico de folds guardado y registrado en MLflow.')

## 9 · Integrated Gradients — Interpretabilidad del mejor modelo

> Implementado con `tf.GradientTape` nativo, sin dependencias externas.

In [ ]:
def integrated_gradients(model, inputs, baseline=None, steps=50):
    """
    Calcula Integrated Gradients para un lote de inputs.
    
    Args:
        model   : modelo Keras compilado
        inputs  : np.array (batch, ...) — observaciones a explicar
        baseline: np.array misma forma que inputs — punto de referencia (0 = promedio nulo)
        steps   : número de pasos de integración (mayor → más preciso)
    Returns:
        ig : np.array misma forma que inputs — atribución de cada feature
    """
    inputs   = tf.cast(inputs,   dtype=tf.float32)
    if baseline is None:
        baseline = tf.zeros_like(inputs)
    baseline = tf.cast(baseline, dtype=tf.float32)

    # Interpolación lineal entre baseline e input
    alphas = tf.linspace(0.0, 1.0, steps + 1)[:, tf.newaxis]  # (steps+1, 1)

    # Para cada input del batch
    all_ig = []
    for idx in range(len(inputs)):
        x       = inputs[idx:idx+1]   # (1, ...)
        base    = baseline[idx:idx+1]

        # Genera path de puntos interpolados
        if x.ndim == 2:
            interp = base + alphas[:, tf.newaxis] * (x - base)   # (steps+1, 1, feat)
            interp = tf.reshape(interp, (-1, x.shape[-1]))
        else:  # 3D (window, feat)
            interp = base + alphas[:, tf.newaxis, tf.newaxis] * (x - base)  # (steps+1, 1, w, f)
            interp = tf.reshape(interp, (-1, x.shape[-2], x.shape[-1]))

        with tf.GradientTape() as tape:
            tape.watch(interp)
            preds = model(interp, training=False)

        grads    = tape.gradient(preds, interp)          # (steps+1, ...)
        avg_grad = tf.reduce_mean(grads, axis=0)         # (...)
        ig       = (x[0] - base[0]) * avg_grad          # (...)
        all_ig.append(ig.numpy())

    return np.array(all_ig)

In [ ]:
# ── Cargar el modelo ganador directamente desde MLflow ────────────────────────
print(f'Cargando modelo ganador: {best_run_name}  (run_id: {best_run_id})')

best_model_uri = f"runs:/{best_run_id}/model"
best_model     = mlflow.tensorflow.load_model(best_model_uri)

# Determinar qué array de entrada usa (secuencial vs lag features)
is_seq_model = (best_run_name != "A_WideDeep")
X_all = X_seq if is_seq_model else X_lag
y_all = y_seq if is_seq_model else y_lag

print(f'Modelo cargado: {best_model.name}')
print(f'Input shape  : {best_model.input_shape}')
print(f'Datos para IG: {X_all.shape}')

In [ ]:
# ── Calcular IG sobre 50 muestras aleatorias ──────────────────────────────────
np.random.seed(42)
sample_idx = np.random.choice(len(X_all), size=50, replace=False)
X_sample   = X_all[sample_idx].astype(np.float32)

ig_values  = integrated_gradients(best_model, X_sample, steps=50)

# Colapsar dimensión temporal si el modelo es secuencial
if ig_values.ndim == 3:
    ig_flat = np.abs(ig_values).mean(axis=1)  # media sobre pasos de tiempo → (50, n_features)
else:
    ig_flat = np.abs(ig_values)[:, :len(FEATURE_COLS)]

mean_ig = ig_flat.mean(axis=0)[:len(FEATURE_COLS)]  # media sobre muestras

# ── Gráfico de importancia ─────────────────────────────────────────────────────
sorted_idx   = np.argsort(mean_ig)[::-1]
top_features = [FEATURE_COLS[i] for i in sorted_idx]
top_values   = mean_ig[sorted_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_features[::-1], top_values[::-1],
        color=COLORS.get(best_run_name, '#607D8B'), edgecolor='white', alpha=0.85)
ax.set_xlabel('Importancia media (|Integrated Gradients|)', fontsize=11)
ax.set_title(f'Integrated Gradients — {best_run_name}\n'
             'Contribución de cada feature a la predicción de NO₃⁻', fontsize=12)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
_ig_plot_path = 'integrated_gradients.png'
plt.savefig(_ig_plot_path, dpi=150, bbox_inches='tight')
plt.show()

# ── Registrar importancias y gráfico en MLflow ────────────────────────────────
ig_dict = {f"ig_{feat}": float(val) for feat, val in zip(FEATURE_COLS, mean_ig)}
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metrics(ig_dict)
    mlflow.log_artifact(_ig_plot_path, artifact_path="interpretability")

print('── Top 5 features más influyentes ──')
for i in range(5):
    print(f'  {i+1}. {top_features[i]:<35s}  IG={top_values[i]:.5f}')
print(f'\nIntegrated Gradients registrados en MLflow run: {best_run_id}')

## 10 · Registro en MLflow Model Registry y exportación de artefactos

In [ ]:
# ── Registrar el modelo ganador en el MLflow Model Registry ───────────────────
REGISTRY_NAME = "nitrate_nowcasting_best"

model_uri       = f"runs:/{best_run_id}/model"
registered_mv   = mlflow.register_model(model_uri, REGISTRY_NAME)

print(f'Modelo registrado en Model Registry:')
print(f'  Nombre  : {registered_mv.name}')
print(f'  Versión : {registered_mv.version}')
print(f'  Run ID  : {best_run_id}')

# ── Guardar escaladores y metadatos como artefactos del run ganador ────────────
os.makedirs('tmp_artifacts', exist_ok=True)

joblib.dump(X_scaler, 'tmp_artifacts/X_scaler.pkl')
joblib.dump(y_scaler, 'tmp_artifacts/y_scaler.pkl')

best_metrics = runs_df.iloc[0]
meta = {
    'model_name':      REGISTRY_NAME,
    'model_version':   registered_mv.version,
    'run_name':        best_run_name,
    'run_id':          best_run_id,
    'window':          WINDOW,
    'lag_steps':       LAG_STEPS,
    'features':        FEATURE_COLS,
    'mae_last3_folds': float(best_metrics['metrics.last3_mae']),
    'r2_last3_folds':  float(best_metrics['metrics.last3_r2']),
    'created':         pd.Timestamp.now().isoformat(),
}
with open('tmp_artifacts/meta.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact('tmp_artifacts/X_scaler.pkl', artifact_path='scalers')
    mlflow.log_artifact('tmp_artifacts/y_scaler.pkl', artifact_path='scalers')
    mlflow.log_artifact('tmp_artifacts/meta.json',    artifact_path='metadata')

print('\n── Artefactos registrados en MLflow ──')
print('  scalers/X_scaler.pkl')
print('  scalers/y_scaler.pkl')
print('  metadata/meta.json')
print()
print('── Metadatos ──')
print(json.dumps(meta, indent=2, ensure_ascii=False))
print()
print('Para cargar el modelo en producción:')
print(f'  model = mlflow.tensorflow.load_model("models:/{REGISTRY_NAME}/latest")')
print('  # o por versión específica:')
print(f'  model = mlflow.tensorflow.load_model("models:/{REGISTRY_NAME}/{registered_mv.version}")')

## 11 · Guía de uso MLflow

### Visualizar experimentos
```bash
# En la terminal, desde la raíz del proyecto:
mlflow ui --port 5000
# → http://127.0.0.1:5000
```

### Comparar runs programáticamente
```python
import mlflow

runs = mlflow.search_runs(
    experiment_names=["nitrate_nowcasting"],
    order_by=["metrics.last3_mae ASC"]
)
print(runs[['tags.mlflow.runName','metrics.last3_mae','metrics.last3_r2']].to_string())
```

### Cargar modelo ganador para inferencia
```python
import mlflow.tensorflow, joblib, numpy as np

# Desde Model Registry (siempre apunta a la mejor versión)
model     = mlflow.tensorflow.load_model("models:/nitrate_nowcasting_best/latest")
X_scaler  = joblib.load("tmp_artifacts/X_scaler.pkl")
y_scaler  = joblib.load("tmp_artifacts/y_scaler.pkl")

# Inferencia (requiere ventana de 8 pasos × 12 features)
X_new     = X_scaler.transform(raw_sensor_data)          # (8, 12)
X_input   = X_new[np.newaxis, ...]                        # (1, 8, 12)
pred_norm = model.predict(X_input).flatten()
pred_mgL  = y_scaler.inverse_transform(pred_norm.reshape(-1, 1)).flatten()
print(f"NO₃⁻ estimado: {pred_mgL[0]:.2f} mg/L")
```

### Flujo completo MLflow
```
tf_experiments.ipynb
  ├── Setup + MLflow Experiment config
  ├── Run A/B/C/D  → mlflow.start_run() por cada arquitectura
  │     ├── mlflow.log_params()          # hiperparámetros
  │     ├── mlflow.log_metrics(step=k)   # MAE/RMSE/R² por fold
  │     ├── mlflow.log_metrics()         # resumen mean/last3
  │     └── mlflow.tensorflow.log_model()# modelo final .tf
  ├── Comparativa  → mlflow.search_runs() ordena por last3_mae
  ├── Gráficos     → mlflow.log_artifact() en run ganador
  ├── Integrated Gradients → mlflow.log_metrics(ig_<feature>)
  └── Model Registry → mlflow.register_model() "nitrate_nowcasting_best"
```